### Dataset info

In [4]:
!ls /kaggle/input/datasets/banuprasadb/visdrone-dataset/VisDrone_Dataset
print("")
!ls /kaggle/input/datasets/banuprasadb/visdrone-dataset/VisDrone_Dataset/VisDrone2019-DET-train

VisDrone2019-DET-test-challenge  VisDrone2019-DET-train  visdrone.yaml
VisDrone2019-DET-test-dev	 VisDrone2019-DET-val

images	labels


### Filtering unnecessary classes

In [5]:
import os
import glob

def filter_yolo_classes(input_dir, output_dir):
    """
    Filters existing YOLO labels.
    Original VisDrone YOLO Classes: 0: pedestrian, 1: people, 2: bicycle, 3: car...
    Our Target Classes: 0: Human (combines 0 & 1), 1: Car (original 3)
    """
    os.makedirs(os.path.join(output_dir, 'images'), exist_ok=True)
    os.makedirs(os.path.join(output_dir, 'labels'), exist_ok=True)
    
    label_paths = glob.glob(os.path.join(input_dir, 'labels', '*.txt'))
    print(f"Processing {len(label_paths)} files in {input_dir}...")
    
    valid_files_count = 0
    
    for label_path in label_paths:
        filename = os.path.basename(label_path)
        img_filename = filename.replace('.txt', '.jpg')
        img_path = os.path.join(input_dir, 'images', img_filename)
        
        if not os.path.exists(img_path): 
            continue
            
        valid_boxes = []
        with open(label_path, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) < 5: continue
                cls_id = int(parts[0])
                
                target_cls = -1
                # Map pedestrian (0) and people (1) to Human (0)
                if cls_id in [0, 1]: 
                    target_cls = 0
                # Map car (3) to Car (1)
                elif cls_id == 3: 
                    target_cls = 1
                    
                if target_cls != -1:
                    # Keep the normalized coordinates exactly as they are
                    valid_boxes.append(f"{target_cls} {' '.join(parts[1:])}\n")
        
        # Only save if the image actually contains a human or a car
        if valid_boxes:
            valid_files_count += 1
            # 1. Write the new filtered label
            with open(os.path.join(output_dir, 'labels', filename), 'w') as f:
                f.writelines(valid_boxes)
            
            # 2. Create a symlink to the image (instant, uses 0 disk space)
            target_img = os.path.join(output_dir, 'images', img_filename)
            if not os.path.exists(target_img):
                os.symlink(img_path, target_img)
                
    print(f"Done! Kept {valid_files_count} images containing humans/cars.")

# Define your paths
base_input = '/kaggle/input/datasets/banuprasadb/visdrone-dataset/VisDrone_Dataset'
base_output = '/kaggle/working/custom_visdrone'

# Run the filter for Train and Val sets
filter_yolo_classes(f'{base_input}/VisDrone2019-DET-train', f'{base_output}/train')
filter_yolo_classes(f'{base_input}/VisDrone2019-DET-val', f'{base_output}/val')

Processing 6471 files in /kaggle/input/datasets/banuprasadb/visdrone-dataset/VisDrone_Dataset/VisDrone2019-DET-train...
Done! Kept 6468 images containing humans/cars.
Processing 548 files in /kaggle/input/datasets/banuprasadb/visdrone-dataset/VisDrone_Dataset/VisDrone2019-DET-val...
Done! Kept 548 images containing humans/cars.


### creating data.yaml for the custom visdrone data

In [6]:
import yaml

data_yaml = {
    'train': '/kaggle/working/custom_visdrone/train/images',
    'val': '/kaggle/working/custom_visdrone/val/images',
    'nc': 2,
    'names': ['Human', 'Car']
}

with open('/kaggle/working/custom_visdrone/data.yaml', 'w') as outfile:
    yaml.dump(data_yaml, outfile, default_flow_style=False)

print("data.yaml created successfully!")

data.yaml created successfully!
